# Publications over the years: data processing

Builds the processed files used by `05b_alba_domains_topics_plots.ipynb` (**run this notebook first**).

| Input (`data/external`) | Description |
|---|---|
| `papers_cleaned.gencat.parquet` | Publicacions de recerca de Catalunya (Dades Obertes Catalunya), cleaned |
| `works.parquet` | OpenAlex metadata of the publications, matched by DOI |
| `fields.parquet` | OpenAlex fields and the domain each field belongs to |

| Output (`data/processed`) | Description |
|---|---|
| `per_years/<year>_publications.parquet` | publications split by year |
| `all_publications_enriched_new.parquet` | one row per DOI, with year, domain and field |
| `all_papers_enriched_covid_months.parquet` | publications of 2019-2022, with year and month (COVID-19 study) |

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Base paths for data
DATA = Path("../data")
EXTERNAL = DATA / "external"
PROCESSED = DATA / "processed"
PER_YEARS = PROCESSED / "per_years"
PER_YEARS.mkdir(parents=True, exist_ok=True) # create folders if they don't exist

## 1. Publications per year

### PRIMARY FILE: papers_cleaned.gencat.parquet

In [2]:
# DATASET: papers_cleaned.gencat.parquet
# Source: Dades obertes de la Generalitat de Catalunya
# Description: cleaned catalan publications
file_path = EXTERNAL / "papers_cleaned.gencat.parquet"

### Load data

In [3]:
def load_local_parquet(path):
    """Load a local parquet file."""
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_parquet(path)

# Load our data from dataset 'papers_cleaned.gencat.parquet'
df = load_local_parquet(file_path)
print(f"Loaded {len(df)} rows.")

Loaded 1747707 rows.


### Checks (and exploration)
We do a check and explore the columns/rows we have in our data.

In [4]:
def profile(df):
    "To avoid error in the cells where there are lists (authors), we convert them into string"
    df_temp = df.copy()
    # Convert all the columns into strings (to avoid errors in lists, arrays...)
    for col in df_temp.columns:
        try:
            df_temp[col] = df_temp[col].astype(str)
        except Exception:
            pass
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df_temp.nunique(),
    })

# Print data profile
display(profile(df))
# Print the first rows to explore the data we have
display(df.head())

,dtype,n_missing,pct_missing,n_unique
date,object,0,0.0,11123
doi,object,0,0.0,358039
ruct,object,0,0.0,88888
orcidResearchers,object,0,0.0,824070
orcidUncheckedAuthors,object,1623199,92.9,15920
orcidDirectors,object,1744810,99.8,64


,date,doi,ruct,orcidResearchers,orcidUncheckedAuthors,orcidDirectors
0,1949-01-01T00:00:00.000,10.1210/jcem-9-9-883,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
1,1952-01-01T00:00:00.000,10.1002/cber.19520850704,Universitat Rovira i Virgili,[0000-0001-6112-5913],None,None
2,1952-01-01T00:00:00.000,10.1111/j.1365-2133.1952.tb15900.x,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
3,1953-01-01T00:00:00.000,10.1038/jid.1953.55,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None
4,1954-01-01T00:00:00.000,10.1016/s0022-5347(17)67795-4,Universitat Rovira i Virgili,[0000-0002-6245-7933],None,None


### Data processing

We separate all the publications in different files per year, and we conserve the following columns: date, DOI (plus the year, extracted from the date).

In [5]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year.astype(int) # from the date, we only want the year

# Order years
anys_disponibles = sorted(df['year'].unique())

# Loop to create a file per year
for any_pub in anys_disponibles:

    # Filter the rows of this year
    df_any = df[df['year'] == any_pub]

    # Create and save file
    nom_fitxer = f"{any_pub}_publications.parquet"
    ruta_final = PER_YEARS / nom_fitxer
    df_any.to_parquet(ruta_final, index=False)
    print(f"Saved: {nom_fitxer} ({len(df_any)} rows)")

print(f"\nDone! {len(anys_disponibles)} files created in '{PER_YEARS.resolve()}'.")

Saved: 1949_publications.parquet (1 rows)
Saved: 1952_publications.parquet (2 rows)
Saved: 1953_publications.parquet (1 rows)
Saved: 1954_publications.parquet (2 rows)
Saved: 1956_publications.parquet (1 rows)
Saved: 1957_publications.parquet (1 rows)
Saved: 1959_publications.parquet (1 rows)
Saved: 1961_publications.parquet (2 rows)
Saved: 1962_publications.parquet (3 rows)
Saved: 1964_publications.parquet (1 rows)
Saved: 1966_publications.parquet (1 rows)
Saved: 1967_publications.parquet (1 rows)
Saved: 1969_publications.parquet (6 rows)
Saved: 1970_publications.parquet (36 rows)
Saved: 1972_publications.parquet (7 rows)
Saved: 1973_publications.parquet (14 rows)
Saved: 1974_publications.parquet (19 rows)
Saved: 1975_publications.parquet (18 rows)
Saved: 1976_publications.parquet (26 rows)
Saved: 1977_publications.parquet (41 rows)
Saved: 1978_publications.parquet (51 rows)
Saved: 1979_publications.parquet (87 rows)
Saved: 1980_publications.parquet (126 rows)
Saved: 1981_publications

## 2. Domains and fields of each publication

### SECONDARY FILES: fields.parquet, works.parquet

We explore the OpenAlex files that give the domain and field of each publication.

In [6]:
# FIELDS (fields.parquet)
fields_df = pd.read_parquet(EXTERNAL / "fields.parquet")

print("Columns:", fields_df.columns.tolist())
print(f"Dimensions: {fields_df.shape[0]} rows and {fields_df.shape[1]} columns")

display(fields_df) # shows all rows (26)

Columns: ['field_id', 'field_name', 'domain_id', 'domain_name']
Dimensions: 26 rows and 4 columns


,field_id,field_name,domain_id,domain_name
0,33,Social Sciences,2,Social Sciences
1,31,Physics and Astronomy,3,Physical Sciences
2,27,Medicine,4,Health Sciences
3,12,Arts and Humanities,2,Social Sciences
4,11,Agricultural and Biological Sciences,1,Life Sciences
5,22,Engineering,3,Physical Sciences
6,17,Computer Science,3,Physical Sciences
7,23,Environmental Science,3,Physical Sciences
8,14,"Business, Management and Accounting",2,Social Sciences
9,13,"Biochemistry, Genetics and Molecular Biology",1,Life Sciences


In [7]:
# WORKS (works.parquet)
works_df = pd.read_parquet(EXTERNAL / "works.parquet")

print(f"Dimensions: {works_df.shape[0]} rows, {works_df.shape[1]} columns")
print("Columns:", works_df.columns.tolist())

display(works_df.head(2))

Dimensions: 348417 rows, 9 columns
Columns: ['openalex_id', 'doi', 'apc_paid_value_usd', 'apc_list_value_usd', 'awards', 'funders', 'primary_topic_field_id', 'topics_field_ids', 'referenced_works']


,openalex_id,doi,apc_paid_value_usd,apc_list_value_usd,awards,funders,primary_topic_field_id,topics_field_ids,referenced_works
0,W2052543964,10.1021/ja01036a083,NaN,4500.0,[],[],16,[],[]
1,W2917211058,10.1016/j.patrec.2019.02.029,NaN,2550.0,[],[],17,[],[]


### Cross data

We join all the per-year files, we keep one row per DOI, and we add to each publication its domain and field (matching the OpenAlex `works` with the `fields`). The result is saved in `all_publications_enriched_new.parquet`.

In [8]:
# Join all the per-year files in one temporary variable called df_total
llista_dfs = [pd.read_parquet(f) for f in PER_YEARS.glob("*_publications.parquet")]
df_total = pd.concat(llista_dfs, ignore_index=True)

# FILTER DUPLICATED DOIS-----------------------------------------------
total_files_originals = len(df_total)
dois_unics_originals = df_total["doi"].nunique()
duplicats_originals = total_files_originals - dois_unics_originals

print(f"Original rows: {total_files_originals}")
print(f"Unique DOIs: {dois_unics_originals}")
print(f"Duplicates detected at the source: {duplicats_originals}")
# ---------------------------------------------------------------------

df_total_unics = df_total.drop_duplicates(subset=["doi"])
print("Unique DOIs (drop duplicates):", len(df_total_unics))

# Load data (fields, domains..)
works = pd.read_parquet(EXTERNAL / "works.parquet", columns=["doi", "primary_topic_field_id", "topics_field_ids"])
fields = pd.read_parquet(EXTERNAL / "fields.parquet")

work_temporal = works.drop_duplicates(subset=["doi"])
total_works_temporal = len(work_temporal)
print("TOTAL WORKS", len(works))
print("Total works without duplicates:", total_works_temporal)
print("Duplicated works:", len(works) - total_works_temporal)

# Force that IDs are text to merge them correctly
works["primary_topic_field_id"] = works["primary_topic_field_id"].astype(str)
fields["field_id"] = fields["field_id"].astype(str)

# Create dictionaries to translate IDs into names
field_to_domain = dict(zip(fields["field_id"], fields["domain_name"]))
field_to_name = dict(zip(fields["field_id"], fields["field_name"]))

# Merge works to the df_total (doi, date)
df_complet = df_total_unics.merge(work_temporal, on="doi", how="left", validate="one_to_one") # merge 'left' to conserve all papers
print("TOTAL MERGED:", len(df_complet))

Original rows: 1747707
Unique DOIs: 358039
Duplicates detected at the source: 1389668
Unique DOIs (drop duplicates): 358039
TOTAL WORKS 348417
Total works without duplicates: 347999
Duplicated works: 418
TOTAL MERGED: 358039


In [9]:
# Add domains and fields names
df_complet["domain_name"] = df_complet["primary_topic_field_id"].map(field_to_domain)
df_complet["field_name"] = df_complet["primary_topic_field_id"].map(field_to_name)

# Select only the columns we want
columnes_finals = ["year", "doi", "domain_name", "field_name", "topics_field_ids"]
df_final = df_complet[columnes_finals]

# Save the file: all_publications_enriched_new.parquet
output_path = PROCESSED / "all_publications_enriched_new.parquet"
df_final.to_parquet(output_path, index=False)

print(f"Dimensions: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
print(f"Columns: {df_final.columns.tolist()}")

# Count papers with and without topic and print nums
total_papers = len(df_final)
papers_sense_topic = df_final["domain_name"].isna().sum()
papers_amb_topic = total_papers - papers_sense_topic

print(f"Total papers: {total_papers}")
print(f"Papers WITHOUT topic assigned: {papers_sense_topic}")
print(f"Papers WITH topic assigned: {papers_amb_topic}")
print(f"Percentage of papers without topic: {(papers_sense_topic / total_papers) * 100:.2f}%")

Dimensions: 358039 rows, 5 columns
Columns: ['year', 'doi', 'domain_name', 'field_name', 'topics_field_ids']
Total papers: 358039
Papers WITHOUT topic assigned: 12313
Papers WITH topic assigned: 345726
Percentage of papers without topic: 3.44%


## 3. COVID-19: publications per month

### Data processing

For the COVID-19 study we keep the publications of 2019-2022 with their year, month, field (`topic_name`) and domain.

In [10]:
df_papers = pd.read_parquet(EXTERNAL / "papers_cleaned.gencat.parquet")

# Convert the column 'date' to a real date and extract year and month
df_papers['date'] = pd.to_datetime(df_papers['date'])
df_papers['year'] = df_papers['date'].dt.year
df_papers['month'] = df_papers['date'].dt.month

# Eliminate duplicated DOIs
print("Publications WITH duplicates:", len(df_papers))
df_unics = df_papers.drop_duplicates(subset=["doi"]).copy()
print("Publications WITHOUT duplicates:", len(df_unics))

# Load data from fields and works
works = pd.read_parquet(EXTERNAL / "works.parquet", columns=["doi", "primary_topic_field_id"])
fields = pd.read_parquet(EXTERNAL / "fields.parquet")

# Eliminate duplicated full rows
print("Works WITH duplicates:", len(works))
works = works.drop_duplicates()
print("Works WITHOUT duplicates:", len(works))

# Prepare merge
fields["field_id"] = fields["field_id"].astype(str)
field_info = fields.set_index("field_id")[["field_name", "domain_name"]].to_dict(orient="index")

# Merge the unique papers with their topics ('left' to conserve all papers)
df_merged = df_unics.merge(works, on="doi", how="left")

# Clean the IDs
df_merged["primary_topic_field_id"] = df_merged["primary_topic_field_id"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Names mapping
df_merged["topic_name"] = df_merged["primary_topic_field_id"].map(lambda x: field_info.get(x, {}).get("field_name"))
df_merged["domain_name"] = df_merged["primary_topic_field_id"].map(lambda x: field_info.get(x, {}).get("domain_name"))

# Filter data from 2019 to 2022
df_covid = df_merged[(df_merged["year"] >= 2019) & (df_merged["year"] <= 2022)].copy()

# Select only the desired columns
columnes_finals = ["year", "month", "doi", "topic_name", "domain_name"]
df_final = df_covid[columnes_finals]

# Save file
output_path = PROCESSED / "all_papers_enriched_covid_months.parquet"
df_final.to_parquet(output_path, index=False)

print(f"Dimensions: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
print("\nFirst rows:")
display(df_final.head())

Publications WITH duplicates: 1747707
Publications WITHOUT duplicates: 358039
Works WITH duplicates: 348417
Works WITHOUT duplicates: 348089
Dimensions: 88150 rows, 5 columns

First rows:


,year,month,doi,topic_name,domain_name
201902,2019,1,10.1093/sleep/zsy205,Psychology,Social Sciences
201903,2019,1,10.1109/tcomm.2018.2876295,Computer Science,Physical Sciences
201904,2019,1,10.1016/j.jchf.2019.04.022,Medicine,Health Sciences
201905,2019,1,10.1016/j.phrs.2018.11.032,"Biochemistry, Genetics and Molecular Biology",Life Sciences
201906,2019,1,10.1111/jsr.12830,Medicine,Health Sciences


### Check

We count the rows without topic/domain. They are kept in the file, and they are shown as `Unknown` in the monthly plot of `05b`.

In [11]:
df = pd.read_parquet(PROCESSED / "all_papers_enriched_covid_months.parquet")

# Count rows
total_files = len(df)
files_amb_topic = df["topic_name"].notna().sum()
files_sense_topic = df["topic_name"].isna().sum()

print(f"Total rows in the file: {total_files}")
print(f"Rows WITH topic: {files_amb_topic}")
print(f"Rows WITHOUT topic (None): {files_sense_topic}")
print(f"Percentage of rows without topic: {(files_sense_topic / total_files) * 100:.2f}%")

# Some examples of papers without topic
print("\nExamples of rows without topic:")
display(df[df["topic_name"].isna()].head(5))

Total rows in the file: 88150
Rows WITH topic: 86032
Rows WITHOUT topic (None): 2118
Percentage of rows without topic: 2.40%

Examples of rows without topic:


,year,month,doi,topic_name,domain_name
89,2019,1,10.5277/abb-01300-2019-02,None,None
116,2019,1,10.3389/fevo.2019.180,None,None
123,2019,1,10.2436/20.1502.01.3,None,None
166,2019,1,10.13137/1825-5167/28382,None,None
188,2019,1,10.4317/jced.55326.ecollection2019jan,None,None


Finally, I noticed that montly data wasn't useful because all papers the date is not known are set on the Jauary 1st of the following year.